# LangChain Tool Conversion Reference

Developer-facing functions defined in `langchain_core.tools.convert`.

# `tool`

Converts a Python function, asynchronous function, or `Runnable` into a LangChain `BaseTool`.

It can be used directly or as a decorator, with or without configuration arguments.

## Supported Calling Forms

### Decorator without arguments

```python
@tool
def function_name(...) -> Any:
    ...
```

The generated tool uses the function's name.

### Decorator with configuration

```python
@tool(
    description=None, # Optional explicit tool description
    return_direct=False, # Whether agent execution stops after this tool
    args_schema=None, # Optional Pydantic model class or JSON schema
    infer_schema=True, # Whether the function signature generates an input schema
    response_format="content", # Expected tool response format
    parse_docstring=False, # Whether Google-style argument descriptions are parsed
    error_on_invalid_docstring=True, # Whether invalid parsed docstrings raise ValueError
    extras=None, # Optional provider-specific tool fields
)
def function_name(...) -> Any:
    ...
```

### Decorator with a custom name

```python
@tool(
    "custom_name", # Name assigned to the generated tool
    description=None, # Optional explicit tool description
    return_direct=False, # Whether agent execution stops after this tool
    args_schema=None, # Optional Pydantic model class or JSON schema
    infer_schema=True, # Whether the function signature generates an input schema
    response_format="content", # Expected tool response format
    parse_docstring=False, # Whether Google-style argument descriptions are parsed
    error_on_invalid_docstring=True, # Whether invalid parsed docstrings raise ValueError
    extras=None, # Optional provider-specific tool fields
)
def function_name(...) -> Any:
    ...
```

### Direct function conversion

```python
generated_tool = tool(
    function, # Python callable converted immediately
    description=None, # Optional explicit tool description
    return_direct=False, # Whether agent execution stops after this tool
    args_schema=None, # Optional Pydantic model class or JSON schema
    infer_schema=True, # Whether the function signature generates an input schema
    response_format="content", # Expected tool response format
    parse_docstring=False, # Whether Google-style argument descriptions are parsed
    error_on_invalid_docstring=True, # Whether invalid parsed docstrings raise ValueError
    extras=None, # Optional provider-specific tool fields
) # Return the generated BaseTool
```

### Direct Runnable conversion

```python
generated_tool = tool(
    "tool_name", # Required string name for Runnable conversion
    runnable, # Runnable converted immediately
    description=None, # Optional explicit tool description
    return_direct=False, # Whether agent execution stops after this tool
    args_schema=None, # Optional Pydantic model class or JSON schema
    infer_schema=True, # Whether an input schema is applied
    response_format="content", # Expected tool response format
    parse_docstring=False, # Whether Google-style descriptions are parsed
    error_on_invalid_docstring=True, # Whether invalid parsed docstrings raise ValueError
    extras=None, # Optional provider-specific tool fields
) # Return the generated BaseTool
```

## Complete Signature

```python
tool(
    name_or_callable: str | Callable[..., Any] | None = None, # Custom name or callable to convert
    runnable: Runnable[Any, Any] | None = None, # Runnable converted directly
    *args: Any, # Must contain no additional positional arguments
    description: str | None = None, # Optional explicit tool description
    return_direct: bool = False, # Whether agent execution stops after this tool
    args_schema: ArgsSchema | None = None, # Optional input-validation schema
    infer_schema: bool = True, # Whether to infer a schema from the callable signature
    response_format: Literal["content", "content_and_artifact"] = "content", # Tool response format
    parse_docstring: bool = False, # Whether to parse Google-style argument descriptions
    error_on_invalid_docstring: bool = True, # Whether invalid parsed docstrings raise ValueError
    extras: dict[str, Any] | None = None, # Provider-specific tool fields
) -> BaseTool | Callable[
    [Callable[..., Any] | Runnable[Any, Any]],
    BaseTool,
] # Return a generated tool or tool-producing decorator
```

## Generated Tool Type

```python
StructuredTool # Generated when schema inference is enabled or args_schema is supplied
Tool # Generated when schema inference is disabled and no args_schema is supplied
```

## Function Conversion Behaviour

- Synchronous functions become synchronously executable tools.
- Asynchronous functions become asynchronously executable tools.
- Function type hints are used for schema inference.
- Functions may contain one or multiple parameters.
- A function name is used as the default tool name.
- A custom positional string overrides the function name.
- A function should provide type hints for accurate schema generation.
- When schema inference is disabled, the function is treated as a simple single-input tool.
- When schema inference is disabled and no schema is supplied, the pinned implementation requires a function docstring.

## Runnable Conversion Behaviour

- Runnable conversion through `tool()` requires an explicit string name.
- The Runnable must expose an object-shaped input schema.
- The Runnable input schema becomes the tool argument schema.
- Tool callbacks are forwarded through the Runnable configuration.
- The Runnable representation becomes the fallback description when no explicit description is supplied.

## Description Precedence

When schema inference is used, the effective description is selected in this order:

```text
1. Explicit description argument
2. Function docstring
3. args_schema description
```

For a Runnable passed to `tool()`, an explicit description is preferred; otherwise, its representation is used.

## Response Formats

### `"content"`

The generated tool's return value is interpreted as tool-message content.

### `"content_and_artifact"`

The wrapped callable must return:

```python
(content, artifact) # Tool-message content and associated artifact
```

## Google-Style Docstring Parsing

Set `parse_docstring=True` to copy function argument descriptions into the generated schema.

```python
@tool(parse_docstring=True)
def repeat_text(text: str, count: int) -> str:
    """Repeat text.

    Args:
        text: Text to repeat.
        count: Number of repetitions.
    """
    return text * count
```

A parsed docstring is invalid when, for example:

- It cannot be separated into a summary and an `Args:` section.
- It documents parameters absent from the function signature.
- Its Google-style structure is malformed.

When `error_on_invalid_docstring=True`, invalid parsed docstrings raise `ValueError`.

## Errors

`tool()` may raise `ValueError` when:

- Extra positional arguments are supplied.
- A Runnable is supplied without a name.
- The name supplied for direct Runnable conversion is not a string.
- The first argument is neither a string nor a callable with `__name__`.
- A Runnable does not expose an object-shaped schema.
- Schema inference is disabled and the required function documentation is absent.
- Docstring parsing is requested and the docstring is invalid.

In [ ]:
from typing_extensions import TypedDict # Import TypedDict for the Runnable input structure

from langchain_core.runnables import RunnableLambda # Import RunnableLambda
from langchain_core.tools import BaseTool, tool # Import the tool converter and base tool type


@tool # Convert the function into a StructuredTool using its original name
def add_numbers(first: int, second: int) -> int: # Define a function with typed arguments
    """Add two numbers.""" # Provide the tool description
    return first + second # Return the addition result


@tool( # Convert the function with custom configuration
    description="Repeat the supplied text a specified number of times", # Set an explicit description
    return_direct=False, # Do not stop agent execution after this tool
    parse_docstring=True, # Parse argument descriptions from the docstring
)
def repeat_text(text: str, count: int) -> str: # Define the configured tool function
    """Repeat text.

    Args:
        text: Text that should be repeated.
        count: Number of times to repeat the text.
    """ # Provide a Google-style docstring
    return text * count # Return the repeated text


@tool("power_calculator") # Convert the function and assign a custom tool name
def calculate_power(number: int, exponent: int) -> int: # Define the power operation
    """Raise a number to a given exponent.""" # Provide the tool description
    return number**exponent # Return the calculated power


def subtract_numbers(first: int, second: int) -> int: # Define a normal Python function
    """Subtract the second number from the first.""" # Provide the tool description
    return first - second # Return the subtraction result


subtract_tool: BaseTool = tool(subtract_numbers) # Convert the function directly into a tool


class MultiplyInput(TypedDict): # Define the object-shaped input required by the Runnable
    first: int # Store the first multiplication value
    second: int # Store the second multiplication value


def multiply_numbers(values: MultiplyInput) -> int: # Define the function used by RunnableLambda
    return values["first"] * values["second"] # Return the multiplication result


multiply_runnable: RunnableLambda = RunnableLambda( # Create a Runnable from the function
    multiply_numbers # Supply the multiplication function
).with_types(input_type=MultiplyInput) # Attach an object-shaped input type


multiply_tool: BaseTool = tool( # Convert the Runnable directly into a tool
    "multiply_numbers", # Assign the required tool name
    multiply_runnable, # Supply the Runnable to convert
    description="Multiply two integer values", # Set the tool description
) # Finish creating the Runnable-based tool


addition_result: int = add_numbers.invoke( # Invoke the decorator-created tool
    {
        "first": 10, # Supply the first number
        "second": 5, # Supply the second number
    }
) # Finish invoking the addition tool

repeat_result: str = repeat_text.invoke( # Invoke the configured tool
    {
        "text": "Hi ", # Supply the text
        "count": 3, # Supply the repetition count
    }
) # Finish invoking the repeat tool

power_result: int = calculate_power.invoke( # Invoke the custom-named tool
    {
        "number": 2, # Supply the base number
        "exponent": 4, # Supply the exponent
    }
) # Finish invoking the power tool

subtraction_result: int = subtract_tool.invoke( # Invoke the directly converted function tool
    {
        "first": 20, # Supply the first number
        "second": 8, # Supply the second number
    }
) # Finish invoking the subtraction tool

multiplication_result: int = multiply_tool.invoke( # Invoke the Runnable-based tool
    {
        "first": 6, # Supply the first multiplication value
        "second": 7, # Supply the second multiplication value
    }
) # Finish invoking the multiplication tool


print("Addition:", addition_result) # Display the addition result

print("Repeated text:", repeat_result) # Display the repeated text

print("Power:", power_result) # Display the power result

print("Subtraction:", subtraction_result) # Display the subtraction result

print("Multiplication:", multiplication_result) # Display the multiplication result

print("Default tool name:", add_numbers.name) # Display the original function-based name

print("Custom tool name:", calculate_power.name) # Display the custom tool name

print("Addition tool type:", type(add_numbers).__name__) # Display the generated tool class

print("Addition arguments:", add_numbers.args) # Display the generated argument schema

# `convert_runnable_to_tool`

Converts an existing `Runnable` into a `BaseTool`.

Unlike `tool()` Runnable conversion, this function supports both string-shaped and object-shaped Runnable inputs.

## Signature

```python
convert_runnable_to_tool(
    runnable: Runnable[Any, Any], # Runnable converted into a tool
    args_schema: TypeBaseModel | None = None, # Optional Pydantic input model
    *,
    name: str | None = None, # Optional tool name
    description: str | None = None, # Optional tool description
    arg_types: dict[str, type] | None = None, # Explicit field names and types used to build a schema
) -> BaseTool # Return the generated tool
```

## Name and Description Defaults

```python
name = name or runnable.get_name() # Use the Runnable name when no tool name is supplied
description = description or generated_description # Use the Runnable input schema in the fallback description
```

The generated fallback description has the form:

```text
Takes {input_json_schema}.
```

## Explicit Input Schema

When `args_schema` is supplied, the Runnable is first wrapped with that input type.

```python
runnable = runnable.with_types(input_type=args_schema)
```

The resulting schema is then used to build the tool.

## String Input Runnable

When the Runnable input JSON schema has type `"string"`, the function returns a simple `Tool`.

```python
Tool(
    name=name, # Tool name
    func=runnable.invoke, # Synchronous Runnable invocation
    coroutine=runnable.ainvoke, # Asynchronous Runnable invocation
    description=description, # Tool description
)
```

## Object Input Runnable

When the Runnable input JSON schema has type `"object"`, the function returns a `StructuredTool`.

Callbacks received by the tool are forwarded to the Runnable configuration.

### Schema Selection

The existing Runnable input schema is reused when:

```python
arg_types is None
and schema["type"] == "object"
and schema contains properties
```

Otherwise, a Pydantic model is generated from `arg_types` or from the Runnable input type hints.

## `arg_types`

`arg_types` explicitly defines the fields accepted by an object-input Runnable tool.

```python
arg_types={
    "first_value": int, # Required integer field
    "second_value": int, # Required integer field
}
```

Each generated field is required.

## Typed Dictionary Inference

When `arg_types` is omitted and a schema must be generated, the function attempts to read type hints from `runnable.InputType`.

Dictionary-like Runnable inputs must therefore be typed, commonly with `TypedDict`.

## Errors

`convert_runnable_to_tool()` may raise `TypeError` when:

- A dictionary-like Runnable input has no usable type annotations.
- Input field types cannot be inferred and `arg_types` is not supplied.

The error can be resolved by either:

```python
runnable = runnable.with_types(input_type=YourTypedDict) # Attach a typed input
```

or:

```python
convert_runnable_to_tool(
    runnable,
    arg_types={"field_name": str},
) # Supply field types explicitly
```

In [ ]:
from typing import Any # Import Any for flexible dictionary values
from typing_extensions import TypedDict # Import TypedDict for typed dictionary input

from pydantic import BaseModel, Field # Import Pydantic classes for an explicit schema

from langchain_core.runnables import RunnableLambda # Import RunnableLambda
from langchain_core.tools import BaseTool # Import the common tool base type
from langchain_core.tools.convert import convert_runnable_to_tool # Import the Runnable conversion function


def uppercase_text(text: str) -> str: # Define a string-input operation
    return text.upper() # Return the uppercase text


string_runnable: RunnableLambda = RunnableLambda(uppercase_text) # Create a string-input Runnable

uppercase_tool: BaseTool = convert_runnable_to_tool( # Convert the string Runnable into a simple Tool
    string_runnable, # Supply the Runnable to convert
    name="uppercase_text", # Assign the tool name
    description="Convert text to uppercase", # Assign the tool description
) # Finish creating the string-input tool


class AddInput(TypedDict): # Define typed object input for the addition Runnable
    first_value: int # Store the first integer
    second_value: int # Store the second integer


def add_values(values: AddInput) -> int: # Define an object-input operation
    return values["first_value"] + values["second_value"] # Return the addition result


add_runnable: RunnableLambda = RunnableLambda(add_values).with_types( # Create a typed object-input Runnable
    input_type=AddInput # Attach the TypedDict input definition
) # Finish configuring the Runnable

add_tool: BaseTool = convert_runnable_to_tool( # Convert the object Runnable into a StructuredTool
    add_runnable, # Supply the typed Runnable
    name="add_values", # Assign the tool name
    description="Add two integer values", # Assign the tool description
) # Finish creating the addition tool


def multiply_values(values: dict[str, int]) -> int: # Define a dictionary-input operation
    return values["first_value"] * values["second_value"] # Return the multiplication result


multiply_runnable: RunnableLambda = RunnableLambda(multiply_values) # Create a dictionary-input Runnable

multiply_tool: BaseTool = convert_runnable_to_tool( # Convert using explicitly supplied argument types
    multiply_runnable, # Supply the Runnable
    name="multiply_values", # Assign the tool name
    description="Multiply two integer values", # Assign the tool description
    arg_types={ # Define the required tool fields explicitly
        "first_value": int, # Require the first integer field
        "second_value": int, # Require the second integer field
    }, # Finish defining the argument types
) # Finish creating the multiplication tool


class DivideInput(BaseModel): # Define an explicit Pydantic input schema
    numerator: float = Field(description="Number to divide") # Define the numerator field
    denominator: float = Field(description="Number used as the divisor") # Define the denominator field


def divide_values(values: DivideInput) -> float: # Define an operation receiving the Pydantic model
    return values.numerator / values.denominator # Return the division result


divide_runnable: RunnableLambda = RunnableLambda(divide_values) # Create the division Runnable

divide_tool: BaseTool = convert_runnable_to_tool( # Convert using an explicit Pydantic schema
    divide_runnable, # Supply the Runnable
    args_schema=DivideInput, # Attach the Pydantic argument schema
    name="divide_values", # Assign the tool name
    description="Divide one number by another", # Assign the tool description
) # Finish creating the division tool


uppercase_result: str = uppercase_tool.invoke("hello langchain") # Invoke the simple string-input Tool

addition_result: int = add_tool.invoke( # Invoke the TypedDict-based StructuredTool
    {
        "first_value": 10, # Supply the first integer
        "second_value": 5, # Supply the second integer
    }
) # Finish invoking the addition tool

multiplication_result: int = multiply_tool.invoke( # Invoke the arg_types-based StructuredTool
    {
        "first_value": 6, # Supply the first integer
        "second_value": 7, # Supply the second integer
    }
) # Finish invoking the multiplication tool

division_result: float = divide_tool.invoke( # Invoke the Pydantic-schema StructuredTool
    {
        "numerator": 20, # Supply the numerator
        "denominator": 4, # Supply the denominator
    }
) # Finish invoking the division tool


print("Uppercase:", uppercase_result) # Display the string-input result

print("Addition:", addition_result) # Display the TypedDict-input result

print("Multiplication:", multiplication_result) # Display the arg_types result

print("Division:", division_result) # Display the Pydantic-schema result

print("String tool type:", type(uppercase_tool).__name__) # Display the generated string tool class

print("Object tool type:", type(add_tool).__name__) # Display the generated object tool class

print("Addition arguments:", add_tool.args) # Display the generated addition schema

## Developer-Facing Top-Level Statements

```python
tool # Decorator and factory that converts functions or Runnables into tools
convert_runnable_to_tool # Convert a Runnable into a Tool or StructuredTool
```

No public classes, constants, exceptions, or type aliases are defined in this module.